# 04 | Group comparison

**No API calls.** Run after all three model notebooks have saved their final results in `reports/`.

All three models predict the same `ppg_3yr` for exactly the same matched players and use the same stratified seed-42 train/test split. A lower test MSE and higher test R² are preferable. Interpret whether college production adds predictive value compared with combine metrics.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
R=Path('reports')
names=['combine','ncaa','combined']
files=[R/f'{n}_metrics.csv' for n in names]
missing=[str(f) for f in files if not f.exists()]
if missing: raise FileNotFoundError('Run the three member notebooks first. Missing: '+', '.join(missing))
results=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
assert results.train_n.nunique()==1 and results.test_n.nunique()==1
results=results.sort_values('test_mse')
display(results[['model','train_n','test_n','test_mse','test_r2','formula']].round(3))
print('Best held-out MSE:',results.iloc[0]['model'])
print('Interpretation: compare whether NCAA and combined features outperform combine-only.')
results.to_csv(R/'group_comparison.csv',index=False)

## Shared exploration plots
Use these figures in the group slides. These charts use the common matched sample.

In [ ]:
# Shared EDA figures for the group presentation. These are descriptions, not model fitting.
import matplotlib.pyplot as plt
features=pd.read_csv('data/predraft_features.csv')
outcomes=pd.read_csv('data/nba_outcomes.csv')
joined=features.merge(outcomes[['player_id','ppg_3yr']],on='player_id').query('ncaa_match_status == "matched"')
fig,ax=plt.subplots(figsize=(6.5,4))
ax.hist(joined.ppg_3yr,bins=22,edgecolor='white')
ax.set(xlabel='NBA PPG over three post-combine seasons',ylabel='Number of players',title='Early-career NBA scoring distribution')
fig.tight_layout();fig.savefig(R/'group_ppg_histogram.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(6.5,4))
for pos,d in joined.groupby('position_group'):
    ax.scatter(d.ncaa_ppg,d.ppg_3yr,label=pos,alpha=.5,s=22)
ax.set(xlabel='Combine-year college-season PPG',ylabel='NBA PPG over three post-combine seasons',title='Does college scoring carry over to the NBA?')
ax.legend();fig.tight_layout();fig.savefig(R/'group_ncaa_vs_nba_scatter.png',dpi=160);plt.show()

### Required discussion

1. Is the NCAA-only model better than the combine-only model?
2. Does the combined model beat NCAA-only on this held-out test sample?
3. Are residual assumptions reasonably satisfied? If not, show **one** documented training-data fix per student.
4. Are missing NCAA records and name-based linkage likely to bias results? **Yes, discuss limitations.**
5. All results refer to players who attended the combine, subsequently played in the NBA, and matched to NCAA records. Do **not** claim they generalize to every NCAA prospect.